# AI광고 vs 인간광고 분류 CNN
EfficientNet-B0 Transfer Learning (과적합 개선 버전)

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
!git clone https://github.com/editpanda-dev/CNN_AIM.git
%cd CNN_AIM

In [ ]:
import os
for cls in ['AI광고', '인간광고']:
    n = len(os.listdir(f'광고_최종/{cls}'))
    print(f'{cls}: {n}장')

In [ ]:
import os, random, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DATA_DIR      = '광고_최종'
CLASS_NAMES   = ['인간광고', 'AI광고']
IMG_SIZE      = 224
BATCH_SIZE    = 32
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 50
LR_HEAD       = 1e-3
LR_FINETUNE   = 1e-4
WEIGHT_DECAY  = 1e-3
VAL_RATIO     = 0.2
SAVE_PATH     = 'best_model.pth'
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
def collect_image_paths(data_dir, class_names):
    EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    paths, labels = [], []
    for label, cls in enumerate(class_names):
        folder = os.path.join(data_dir, cls)
        for fname in os.listdir(folder):
            if os.path.splitext(fname)[1].lower() in EXTS:
                paths.append(os.path.join(folder, fname))
                labels.append(label)
    return paths, labels

class AdDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE+48, IMG_SIZE+48)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomRotation(20),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

In [ ]:
def build_model(num_classes=2, dropout=0.5):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, 256),
        nn.SiLU(),
        nn.Dropout(dropout / 2),
        nn.Linear(256, num_classes),
    )
    return model

def freeze_backbone(model):
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False

# 마지막 3개 블록 + classifier만 unfreeze (소량 데이터 핵심)
def partial_unfreeze(model):
    for param in model.parameters():
        param.requires_grad = False
    # features[6], features[7], features[8] + classifier
    for name, param in model.named_parameters():
        if any(f'features.{i}' in name for i in [6, 7, 8]) or 'classifier' in name:
            param.requires_grad = True

def mixup(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return x_mix, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def run_epoch(model, loader, criterion, optimizer=None, phase='train'):
    is_train = phase == 'train'
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if is_train:
                imgs, y_a, y_b, lam = mixup(imgs, labels)
                outputs = model(imgs)
                loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
            else:
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

In [ ]:
all_paths, all_labels = collect_image_paths(DATA_DIR, CLASS_NAMES)
print(f'전체: {len(all_paths)} (인간:{all_labels.count(0)}, AI:{all_labels.count(1)})')

tr_paths, val_paths, tr_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=VAL_RATIO, random_state=SEED, stratify=all_labels)
print(f'Train: {len(tr_paths)}, Val: {len(val_paths)}')

class_counts = [tr_labels.count(i) for i in range(2)]
sample_weights = [1.0 / class_counts[l] for l in tr_labels]
sampler = WeightedRandomSampler(sample_weights, len(tr_labels), replacement=True)

train_loader = DataLoader(AdDataset(tr_paths, tr_labels, train_tf),
                          batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(AdDataset(val_paths, val_labels, val_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
model     = build_model().to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
best_val_acc = 0.0

# ── Phase 1: Head만 ──────────────────────────────────
print('=== Phase 1: Head만 학습 ===')
freeze_backbone(model)
opt1 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                   lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
sch1 = optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS)

for ep in range(1, PHASE1_EPOCHS+1):
    tl, ta = run_epoch(model, train_loader, criterion, opt1, 'train')
    vl, va = run_epoch(model, val_loader,   criterion, None, 'val')
    sch1.step()
    marker = ''
    if va > best_val_acc:
        best_val_acc = va; torch.save(model.state_dict(), SAVE_PATH); marker = '  ← best'
    print(f'[P1 {ep:02d}/{PHASE1_EPOCHS}] train={ta:.4f} | val={va:.4f}{marker}')

In [ ]:
# ── Phase 2: 마지막 3블록만 unfreeze ────────────────
print('\n=== Phase 2: Partial Fine-tune (마지막 3블록) ===')
partial_unfreeze(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'학습 파라미터: {trainable:,}개')

opt2 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                   lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
sch2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS, eta_min=1e-6)
patience, no_improve = 15, 0

for ep in range(1, PHASE2_EPOCHS+1):
    tl, ta = run_epoch(model, train_loader, criterion, opt2, 'train')
    vl, va = run_epoch(model, val_loader,   criterion, None, 'val')
    sch2.step()
    marker = ''
    if va > best_val_acc:
        best_val_acc = va; torch.save(model.state_dict(), SAVE_PATH); marker = '  ← best'; no_improve = 0
    else:
        no_improve += 1
    print(f'[P2 {ep:02d}/{PHASE2_EPOCHS}] train={ta:.4f} | val={va:.4f}{marker}')
    if no_improve >= patience:
        print('Early stopping'); break

print(f'\n최고 Val Accuracy: {best_val_acc*100:.1f}%')
print(f'모델 저장: {SAVE_PATH}')